In [2]:
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
DATA_DIR = Path("20260624")
files = sorted(glob.glob(str(DATA_DIR / "*.xlsx")))

data_list = []
for file in files:
    print(f"Read file {file}")
    temp = pd.read_excel(file, skiprows=7)
    temp["source_file"] = Path(file).name
    data_list.append(temp)
    print(temp.shape)

df_raw = pd.concat(data_list, ignore_index=True)

line_items = sorted(
    df_raw.loc[df_raw["Line Item"].notna() & (df_raw["Line Item"] != "Totals"), "Line Item"].unique()
)

settings = {}
for line_item in line_items:
    name = line_item.lower()
    if "ret_" in name:
        category = "RTG"
    elif "per_" in name:
        category = "Seed"
    elif "kids" in name:
        category = "Kids"
    elif "body" in name:
        category = "Body"
    else:
        category = "Other"
    settings.setdefault(category, []).append(line_item)

campaign_column = "Creative Group"
creative_groups = sorted(df_raw.loc[df_raw[campaign_column].notna(), campaign_column].unique())
settings_by_creative = {}
for creative_group in creative_groups:
    category = creative_group.split(" - ")[2].split()[0]
    settings_by_creative.setdefault(category, []).append(creative_group)

print("Unique line items:", line_items)
print("Settings:", settings)
print("Settings by creative group:", settings_by_creative)

In [4]:
cost_column = "Costs (Total)"
revenue_column = "Revenue (Original, DDA Enhanced, ToCnv)"


def parse_numeric(series):
    if series.dtype == object:
        return (
            series.astype(str)
            .str.replace("\xa0", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
            .replace({"nan": None})
            .astype(float)
        )
    return pd.to_numeric(series, errors="coerce")


def parse_cdr(series):
    cdr = parse_numeric(series)
    # Excel exports store percentage-formatted CDR as ratios (0.1616 = 16.16%)
    if cdr.max(skipna=True) <= 10:
        cdr = cdr * 100
    return cdr


def prepare_numeric_columns(frame):
    frame = frame.copy()
    frame[cost_column] = parse_numeric(frame[cost_column])
    frame[revenue_column] = parse_numeric(frame[revenue_column])
    frame["CDR"] = parse_cdr(frame["Tchibo CDR Net"])
    return frame


df = prepare_numeric_columns(
    df_raw[df_raw[campaign_column].notna() & df_raw["Week"].notna()]
).reset_index(drop=True)
df = df.sort_values(by=[campaign_column, "Week"], ascending=True).reset_index(drop=True)

df_line_items = prepare_numeric_columns(
    df_raw[
        df_raw["Line Item"].notna()
        & df_raw["Week"].notna()
        & df_raw[campaign_column].isna()
        & (df_raw["Line Item"] != "Totals")
    ]
).reset_index(drop=True)
df_line_items = df_line_items.sort_values(by=["Line Item", "Week"], ascending=True).reset_index(drop=True)
df["ROAS"] = df[revenue_column] / df[cost_column]
df[[campaign_column, "Week", "Tchibo CDR Net", cost_column, revenue_column, "CDR", "ROAS", "source_file"]]

In [5]:
campaigns = df[campaign_column].unique()
line_items

In [6]:
settings_by_creative

In [7]:
key = "RTG"
temp = df[df[campaign_column].isin(settings_by_creative[key])]
fig = plt.figure(figsize=(12, 16))
ax = fig.add_subplot(411)
ax.set_title("Costs")
sns.lineplot(data=temp, x="Week", y=cost_column, hue=campaign_column, marker="o", ax=ax, errorbar=None)
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True)
ax2 = fig.add_subplot(412)
ax2.set_title("Revenue")
sns.lineplot(data=temp, x="Week", y=revenue_column, hue=campaign_column, marker="o", ax=ax2, errorbar=None)
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True)
ax3 = fig.add_subplot(413)
ax3.set_title("Tchibo CDR Net")
sns.lineplot(data=temp, x="Week", y="CDR", hue=campaign_column, marker="o", ax=ax3, errorbar=None)
plt.xticks(rotation=90)
max_ylim = temp["CDR"].max()
plt.ylim(0, max_ylim)
plt.tight_layout()
plt.grid(True)
ax4 = fig.add_subplot(414)
ax4.set_title("ROAS")
sns.lineplot(data=temp, x="Week", y="ROAS", hue=campaign_column, marker="o", ax=ax4, errorbar=None)
ax4.axhline(1, color="gray", linestyle="--", linewidth=1)
plt.xticks(rotation=90)
plt.ylim(0, temp["ROAS"].max())
plt.tight_layout()
plt.grid(True)
plt.show()


In [10]:
temp.groupby(campaign_column).agg({"CDR":["min", "max", "mean", "median", "std"]})

In [9]:
key = "Seed"
temp_seed = df[df[campaign_column].isin(settings_by_creative[key])]
fig = plt.figure(figsize=(12, 16))
ax = fig.add_subplot(411)
ax.set_title("Costs")
sns.lineplot(data=temp_seed, x="Week", y=cost_column, hue=campaign_column, marker="o", ax=ax, errorbar=None)
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True)
ax2 = fig.add_subplot(412)
ax2.set_title("Revenue")
sns.lineplot(data=temp_seed, x="Week", y=revenue_column, hue=campaign_column, marker="o", ax=ax2, errorbar=None)
plt.xticks(rotation=90)
plt.tight_layout()
plt.grid(True)
ax3 = fig.add_subplot(413)
ax3.set_title("Tchibo CDR Net")
sns.lineplot(data=temp_seed, x="Week", y="CDR", hue=campaign_column, marker="o", ax=ax3, errorbar=None)
plt.xticks(rotation=90)
plt.ylim(0, temp_seed["CDR"].max())
plt.tight_layout()
plt.grid(True)
ax4 = fig.add_subplot(414)
ax4.set_title("ROAS")
sns.lineplot(data=temp_seed, x="Week", y="ROAS", hue=campaign_column, marker="o", ax=ax4, errorbar=None)
ax4.axhline(1, color="gray", linestyle="--", linewidth=1)
plt.xticks(rotation=90)
plt.ylim(0, temp_seed["ROAS"].max())
plt.tight_layout()
plt.grid(True)
plt.show()
